# 03 Information leakage: A vs B

A decomposes the complete 404-day series and therefore uses future observations. B decomposes only the 279-day training segment and extrapolates its IMFs with ARIMA.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.volatility import load_rv_series, validate_split
from src.decomposition import select_first_k
from src.evaluation import (
    decompose_full_and_fragment,
    reconstruction_rmse,
)

FIG_DIR = REPO_ROOT / "results" / "figures"
TABLE_DIR = REPO_ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["axes.unicode_minus"] = False

## Load data and split at 2024-02-29

In [ ]:
rv, source = load_rv_series(REPO_ROOT / "data" / "rv_series.csv", allow_network=False)
train_end_idx, n_test = validate_split(rv)
print("source:", source)
print("train end index:", train_end_idx)
print("test length:", n_test)
print("test start:", rv.index[train_end_idx].date())

## Rerun A and B inside the same evaluation code

In [ ]:
baseline = decompose_full_and_fragment(
    rv,
    train_end_idx,
    n_test=n_test,
    trials=100,
    parallel=False,
    seed=42,
)
print("full elapsed:", baseline["full_elapsed_sec"])
print("fragment elapsed:", baseline["fragment_elapsed_sec"])

## k=1, 3, 5 reconstruction errors

In [ ]:
truth = rv.values[train_end_idx:]
full_test = baseline["imfs_full"][:, train_end_idx:]
fragment_test = baseline["imfs_fragment_extrapolated"][:, :n_test]

rows = []
for k in (1, 3, 5):
    recon_full = np.sum(select_first_k(full_test, k), axis=0)
    recon_fragment = np.sum(select_first_k(fragment_test, k), axis=0)
    rows.append(
        {
            "k": k,
            "full_sample_leaky_rmse": reconstruction_rmse(truth, recon_full),
            "fragment_arima_rmse": reconstruction_rmse(truth, recon_fragment),
        }
    )
leakage_table = pd.DataFrame(rows)
leakage_table["leakage_gain_pct"] = (
    (leakage_table["fragment_arima_rmse"] - leakage_table["full_sample_leaky_rmse"])
    / leakage_table["fragment_arima_rmse"]
    * 100.0
)
leakage_table.to_csv(TABLE_DIR / "leakage_comparison.csv", index=False)
leakage_table

## Reconstruction curves

In [ ]:
dates = rv.index[train_end_idx:]
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
for ax, k in zip(axes, (1, 3, 5)):
    full_recon = np.sum(select_first_k(full_test, k), axis=0)
    fragment_recon = np.sum(select_first_k(fragment_test, k), axis=0)
    ax.plot(dates, truth, color="black", linewidth=1.7, label="True RV")
    ax.plot(dates, full_recon, color="red", linewidth=1.1, label="A: full-sample (leaky)")
    ax.plot(dates, fragment_recon, color="blue", linewidth=1.1, label="B: fragment + ARIMA")
    ax.set_title(f"Information leakage comparison, k={k}")
    ax.set_ylabel("GK variance")
    ax.grid(alpha=0.25)
    ax.legend()
axes[-1].set_xlabel("Date")
fig.tight_layout()
fig.savefig(FIG_DIR / "leakage_comparison.png", dpi=200, bbox_inches="tight")
plt.show()